In [41]:
import re, unicodedata
from pathlib import Path
from collections import Counter

from src.utils.helpers import is_chinese

In [ ]:
LABEL_FILE  = 'D://dataset/HME100K/train.txt'
OUTPUT_FILE = './dataset/processed/train_clean.txt'

NOISE_TOKENS = {'"', "'", '|', '…', '—', '–'}

In [43]:
NORMALIZE = {
    "、": ',', 
    "。": '.',
    '√':  '\\sqrt',
    '²':  '^{2}',      # superscript 2
    '³':  '^{3}',
    '×':  '\\times',
    '÷':  '\\div',
    '±':  '\\pm',
    '≤':  '\\leq',
    '≥':  '\\geq',
    '≠':  '\\neq',
    '≈':  '\\approx',
    '∞':  '\\infty',
    '→':  '\\rightarrow',
    '←':  '\\leftarrow',
    '∈':  '\\in',
    '∉':  '\\notin',
    '∩':  '\\cap',
    '∪':  '\\cup',
    '∑':  '\\sum',
    '∏':  '\\prod',
    '∫':  '\\int',
    'π':  '\\pi',
    'α':  '\\alpha',
    'β':  '\\beta',
    'γ':  '\\gamma',
    'θ':  '\\theta',
    'λ':  '\\lambda',
    'μ':  '\\mu',
    'σ':  '\\sigma',
    'φ':  '\\phi',
    'ω':  '\\omega',
}

def normalize_tokens(toks: list) -> list:
    return [NORMALIZE.get(t, t) for t in toks]


In [44]:
raw_samples = []
with open(LABEL_FILE, encoding='utf-8') as f:
    for lineno, line in enumerate(f, 1):
        line = line.rstrip('\n')
        if not line.strip() or '\t' not in line:
            continue
        img, latex = line.split('\t', 1)
        toks = normalize_tokens(latex.strip().split())   # ← normalize đây
        raw_samples.append({'img': img.strip(), 'toks': toks,
                            'latex': ' '.join(toks)})

print(f'Tổng mẫu ban đầu: {len(raw_samples):,}')

Tổng mẫu ban đầu: 99,109


In [45]:
all_toks  = [t for s in raw_samples for t in s['toks']]
tok_freq  = Counter(all_toks)

chinese_toks  = {t for t in tok_freq if is_chinese(t)}
noise_toks    = {t for t in tok_freq if t in NOISE_TOKENS}

print('\nToken tiếng Trung:')
for t in sorted(chinese_toks):
    print(f'{t:15s}  count={tok_freq[t]}')

print(f'\nToken noise khác:')
for t in sorted(noise_toks):
    print(f'{t!r:15s}  count={tok_freq[t]}')


Token tiếng Trung:
丙                count=1
个                count=3
为                count=1
乙                count=1
人                count=1
代                count=2
以                count=1
倍                count=1
元                count=2
入                count=2
公                count=1
分                count=1
即                count=2
原                count=1
可                count=1
周                count=1
和                count=1
因                count=5
大                count=2
天                count=1
妈                count=1
将                count=1
小                count=1
岁                count=1
年                count=1
式                count=1
得                count=1
或                count=2
所                count=1
把                count=1
数                count=7
方                count=1
时                count=2
明                count=1
是                count=3
最                count=1
本                count=1
次                count=2
求                count=1
法    

In [46]:
print(f'\nCặp token khác nhau hoa/thường (có thể noise):')
lower_map = {}   # lowercase → list of original forms
for t in tok_freq:
    key = t.lower()
    lower_map.setdefault(key, []).append(t)
ambiguous = {k: v for k, v in lower_map.items() if len(v) > 1}
if ambiguous:
    for key, variants in sorted(ambiguous.items()):
        counts = {v: tok_freq[v] for v in variants}
        # Chỉ cảnh báo nếu ít nhất 1 variant rất hiếm (có thể typo)
        max_c  = max(counts.values())
        min_c  = min(counts.values())
        flag   = '  ← có thể typo' if min_c <= 5 and max_c > 10 else ''
        print(f'      {variants}  counts={list(counts.values())}{flag}')
else:
    print('(không có)')


Cặp token khác nhau hoa/thường (có thể noise):
      ['\\Delta', '\\delta']  counts=[2987, 1]  ← có thể typo
      ['\\Gamma', '\\gamma']  counts=[2, 5]
      ['\\lambda', '\\Lambda']  counts=[199, 8]
      ['\\leftarrow', '\\Leftarrow']  counts=[6, 1]
      ['\\Omega', '\\omega']  counts=[693, 7]
      ['\\phi', '\\Phi']  counts=[11, 2]  ← có thể typo
      ['\\rightarrow', '\\Rightarrow']  counts=[322, 419]
      ['a', 'A']  counts=[19426, 18290]
      ['B', 'b']  counts=[16080, 8946]
      ['C', 'c']  counts=[19725, 7109]
      ['D', 'd']  counts=[10690, 1699]
      ['E', 'e']  counts=[7387, 1345]
      ['F', 'f']  counts=[6054, 945]
      ['g', 'G']  counts=[2769, 1918]
      ['H', 'h']  counts=[6505, 1109]
      ['I', 'i']  counts=[1348, 397]
      ['J', 'j']  counts=[276, 25]
      ['k', 'K']  counts=[5095, 284]
      ['l', 'L']  counts=[2721, 615]
      ['m', 'M']  counts=[12473, 2034]
      ['N', 'n']  counts=[4337, 6079]
      ['O', 'o']  counts=[12885, 706]
      ['P', 'p'] 

In [47]:
bad_tokens = chinese_toks | noise_toks   # xóa cả 2

removed_chinese = []
removed_noise   = []
clean_samples   = []

for s in raw_samples:
    tok_set = set(s['toks'])
    if tok_set & chinese_toks:
        removed_chinese.append(s)
    elif tok_set & noise_toks:
        removed_noise.append(s)
    else:
        clean_samples.append(s)

print(f'Kết quả lọc:')
print(f'Xóa (có tiếng Trung): {len(removed_chinese):>6,}')
print(f'Xóa (noise token): {len(removed_noise):>6,}')
print(f'Giữ lại: {len(clean_samples):>6,}')
print(f'Tỷ lệ giữ lại: {len(clean_samples)/len(raw_samples)*100:.2f}%')

Kết quả lọc:
Xóa (có tiếng Trung):     55
Xóa (noise token):      2
Giữ lại: 99,052
Tỷ lệ giữ lại: 99.94%


In [48]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for s in clean_samples:
        f.write(f"{s['img']}\t{s['latex']}\n")